## IoT Motion Data Analysis

### Data Loading
This section loads the dataset from a PostgreSQL database using SQL queries and prepares it for analysis in Python.

In [ ]:
!pip install pandas psycopg2-binary

In [ ]:
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="iot_motion_project",
    user="postgres",
    password="password"
)

query = "SELECT * FROM motion_data"

df = pd.read_sql(query, conn)

df.head()

### Data Overview
This section provides an overview of the dataset structure, similar to initial SQL exploration, including columns, data types, and data completeness.

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
!pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

### Activity Distribution Analysis
This section analyzes the distribution of walking and running activities, corresponding to the SQL GROUP BY activity query.

In [ ]:
activity_counts = df['activity'].value_counts()

In [ ]:
plt.bar(activity_counts.index, activity_counts.values)

plt.title("Activity Distribution")
plt.xlabel("Activity (0 = Walking, 1 = Running)")
plt.ylabel("Count")

plt.savefig("Desktop/iot_motion/iot_motion_images/activity_distribution.png")
plt.show()

### Magnitude Distribution Analysis (Feature Engineering)
This section visualizes the distribution of acceleration magnitude for each activity, supporting the insights obtained from SQL aggregation queries.

In [ ]:
df['magnitude'] = (df['acc_x']**2 + df['acc_y']**2 + df['acc_z']**2)**0.5

In [ ]:
plt.hist(df[df['activity'] == 0]['magnitude'], bins=50, alpha=0.5, label='Walking')
plt.hist(df[df['activity'] == 1]['magnitude'], bins=50, alpha=0.5, label='Running')

plt.title("Acceleration Magnitude Distribution")
plt.xlabel("Magnitude")
plt.ylabel("Frequency")
plt.legend()


plt.savefig("iot_motion_images/iot_magnitude_distribution.png")

plt.show()

In [ ]:
df.groupby('activity')['magnitude'].agg(['min', 'max'])

In [ ]:
!pip install seaborn

In [ ]:
import os
os.listdir()

## Correlation Analysis

This section analyzes the relationships between accelerometer, gyroscope, and magnitude features to identify which variables may be most useful for activity classification.

In [ ]:
corr = df.corr(numeric_only=True)
corr

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,7))
sns.heatmap(corr, annot=False, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.savefig("Desktop/iot_motion/iot_motion_images/correlation.png")
plt.show()

## Magnitude Comparison by Activity

This boxplot compares acceleration magnitude between walking and running activities to visualize differences in motion intensity and variability.

In [ ]:
df.columns

In [ ]:
df['magnitude'] = (
    (df['acc_x']**2 +
     df['acc_y']**2 +
     df['acc_z']**2) ** 0.5
)

In [ ]:
sns.boxplot(x='activity', y='magnitude', data=df)

plt.title("Magnitude by Activity")
plt.xlabel("Activity (0 = Walking, 1 = Running)")
plt.ylabel("Acceleration Magnitude")

plt.savefig("Desktop/iot_motion/iot_motion_images/magnitude_boxplot.png")
plt.show()

## Key EDA Insights

- `acc_y` shows the strongest relationship with activity classification, indicating it may be one of the most important features for distinguishing walking and running.

- Running activity (activity = 1) has a wider and higher acceleration magnitude distribution compared to walking, suggesting greater motion intensity and variability.

- Acceleration magnitude remains a strong engineered feature for activity classification and is expected to be important in machine learning models.